# 🛰️ OrbitalDelta - GPU Training

Run this notebook in **Google Colab** or **Kaggle** with a GPU enabled (T4, P100, etc.) to train the Siamese U-Net model on the LEVIR-CD dataset.

## 1. Setup Environment & Clone Repository
First, we need to clone your codebase and install the required dependencies. Replace the `<YOUR_GITHUB_REPO_URL>` placeholder with the actual URL.

In [ ]:
!git clone <YOUR_GITHUB_REPO_URL> OrbitalDelta
%cd OrbitalDelta
!pip install -e .

## 2. Download the LEVIR-CD Dataset
This script will pull the LEVIR-CD dataset into `data/raw/`.

In [ ]:
!python -m src.data.download --dataset levir-cd --output data/raw/

## 3. Preprocess the Dataset
This crops the 1024x1024 images into 256x256 patches and splits them into train, val, and test sets. **This might take a few minutes.**

In [ ]:
!python -m src.data.preprocess --input data/raw/levir-cd --output data/processed/levir-cd --crop-size 256

## 4. Train the Model
Now we start the actual GPU training. The config `configs/train_levir.yaml` dictates batch size, epochs, and early stopping. It will automatically use the GPU if available.

In [ ]:
!python scripts/train.py --config configs/train_levir.yaml

## 5. Evaluate the Model
Once training is done or stopped early, evaluate the best checkpoint on the test set to ensure we hit our F1 >= 0.88 target.

In [ ]:
!python scripts/evaluate.py --checkpoint checkpoints/best.pt --config configs/train_levir.yaml

## 6. Generate Visualizations (Optional)
Let's generate some image outputs (A, B, Ground Truth, Prediction) to see how the model performs visually.

In [ ]:
!python scripts/visualize.py --checkpoint checkpoints/best.pt --config configs/train_levir.yaml --num-samples 25

# Zip the outputs to download them
!zip -r outputs.zip outputs/visualizations/
!zip -r checkpoints.zip checkpoints/

## 7. Download Your Weights
Run this cell to download the trained weights back to your local machine (if using Google Colab). You will place this `best.pt` file inside your local `checkpoints/` folder.

In [ ]:
try:
    from google.colab import files
    files.download('checkpoints/best.pt')
    files.download('outputs.zip')
except ImportError:
    print("Not running in Google Colab. Please download the 'checkpoints/best.pt' file manually via the file browser!")